# SHAP — Explain a vol forecaster

## Project goal

Take a fitted LightGBM vol forecaster and audit it with SHAP. Produce a beeswarm summary, dependence plots for the top features, a waterfall for the most extreme prediction, top-pair interaction values, and a 'top-K refit' ablation showing whether you can drop the bottom half of features without loss.


## Why this exercises the cheatsheet

SHAP is most useful when interrogating a fitted model — distinguishing 'works because of real signal' from 'works because of a leak'. This project exercises the new Explanation API end to end, including the slicing and interaction-value patterns from the cheatsheet.


## Sub-task 1: Train (or load) the model

Train a LightGBM regressor on BTC features → next-24h vol target (or load the bundle from `gradient_boosting_project` if you've done it). Subsample val to 1000-3000 rows for SHAP (full val is slow).

**Patterns this forces:**

- reuse from prior project OR fresh fit
- subsample for speed: `X_val_sample = X_val.sample(n=2000, random_state=0)`


In [ ]:
# Your answer here

## Sub-task 2: Compute SHAP via TreeExplainer

Use the new Explanation API: `explainer = shap.Explainer(model, X_train)`; `exp = explainer(X_val_sample)`. Inspect `exp.values.shape`, `exp.base_values[:3]`, `exp.feature_names`.

**Patterns this forces:**

- `shap.Explainer(model, X_train)` (auto-picks TreeExplainer)
- `exp = explainer(X_val_sample)`
- the new Explanation object's API


In [ ]:
# Your answer here

## Sub-task 3: Beeswarm summary

`shap.plots.beeswarm(exp, max_display=10)`. Read the plot: which features have the widest impact range? Which show clear monotonic colour gradients (= linear effect)?

**Patterns this forces:**

- `shap.plots.beeswarm(exp, max_display=...)`
- interpretation: spread vs colour-gradient


In [ ]:
# Your answer here

## Sub-task 4: Dependence plots for the top 2 features

Identify the top-2 features by mean |SHAP|. For each, `shap.dependence_plot('feat_name', exp.values, X_val_sample)`. Comment on the shape: linear, threshold, or interactive?

**Patterns this forces:**

- `mean_abs = np.abs(exp.values).mean(axis=0)`
- `np.argsort(mean_abs)[::-1][:2]`
- `shap.dependence_plot(...)`


In [ ]:
# Your answer here

## Sub-task 5: Waterfall on the most extreme prediction

Find the row in `X_val_sample` with the highest predicted vol. Show `shap.plots.waterfall(exp[i], max_display=8)`. Interpret in plain English: which features pushed this prediction up the most?

**Patterns this forces:**

- `i = int(np.argmax(model.predict(X_val_sample)))`
- `shap.plots.waterfall(exp[i])`
- the new Explanation slicing: `exp[i]` is one row's Explanation


In [ ]:
# Your answer here

## Sub-task 6: Slice the Explanation by feature value

Use `exp[:, 'vol_24h_trailing']` to get all SHAP values for that feature. Filter rows where the feature value is in the top decile of `X_val_sample['vol_24h_trailing']`. Compute mean SHAP for that slice. Question: in high-vol regimes, how does the model's reliance on this feature change?

**Patterns this forces:**

- `exp[:, 'feat_name']` slicing the Explanation
- boolean mask on the underlying X to subset
- mean SHAP per slice


In [ ]:
# Your answer here

## Sub-task 7: Interaction values, top pair

Compute `explainer.shap_interaction_values(X_val_sample.iloc[:500])`. Aggregate to a feature × feature matrix of mean |interaction|. Identify the pair with the highest off-diagonal value. Plot a 2D scatter of the two features' values, coloured by their interaction SHAP value.

**Patterns this forces:**

- `explainer.shap_interaction_values(X_subsample)`
- off-diagonal extraction
- 2D scatter coloured by interaction strength


In [ ]:
# Your answer here

## Sub-task 8: Top-K ablation

Take the top 50% of features by mean |SHAP|. Refit a fresh LightGBM with the SAME hyperparameters but only those features. Compare val MAE to the full-feature model. Comment: how much performance does dropping the bottom half cost?

**Patterns this forces:**

- `top_k = features[np.argsort(mean_abs)[::-1][:k]]`
- `clone(model).fit(X_train[top_k], y_train)`
- MAE comparison


In [ ]:
# Your answer here

## What success looks like

- One beeswarm plot, two dependence plots, one waterfall.
- A 2D interaction scatter showing a clear coloured pattern (= real interaction) or no clear pattern (= just main effects).
- The top-K refit either matches or comes within 5% of the full-feature MAE.
- A short paragraph: *'the model relies primarily on (top-3 features); the biggest interaction is (X, Y); the bottom half of features are essentially decorative.'*
